In [ ]:
import sys
import time
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
import struct

FLASH_BASE = 0x08000000
FLASH_SIZE = 512 * 1024      # STM32L452RE: 512 KB


def set_debug_pin(value):
    if value:
        scope.io.tio3 = "gpio_high"
    else:
        scope.io.tio3 = "gpio_low"
        
def rx(n, timeout=300):
    """Read up to n bytes; returns fewer on timeout."""
    data = ser.read(n, timeout=timeout).encode("latin-1")
    if debug:
        print("rx: ", data.hex(" "))
    return data

def tx(data, write_single_bytes=False):
    if debug:
        print("tx: ", data.hex(" "))

    if write_single_bytes:
        for b in data:
            ser.write(bytes([b]))
            if BYTE_GAP:
                time.sleep(BYTE_GAP)
    else:
        ser.write(data)


def enter_bootloader():
    """BOOT0 high, reset, then send 0x7F for auto-baud."""
    for _ in range(RETRIES):
        if debug:
            print("enter_bootloader() called")  # temporary debug
        scope.io.pdic = "high"
        time.sleep(0.05)
        scope.io.nrst = "low"
        time.sleep(0.2)
        scope.io.nrst = "high_z"
        time.sleep(0.5)
        ser.flush()
        tx(b"\x7f")
        reply = rx(1)
        if reply == b"\x79":
            return
        print(f"  handshake reply: {reply.hex() or '(none)'}")
    raise RuntimeError("no ACK to 0x7F handshake")


def cmd(code):
    ser.flush()                       # resync: drop leftovers
    tx(bytes([code, code ^ 0xFF]))
    return rx(1)


def read_mem(addr, length):
    """Read Memory command (0x11), length 1..256."""
    set_debug_pin(False)
    # send 0x11 0xEE + wait for ACK
    if cmd(0x11) != b"\x79":
        raise RuntimeError("Read Memory NACK")

    # send address MSB ... LSB
    # send xx xx xx xx xor + wait for ACK
    a = addr.to_bytes(4, "big")
    tx(a + bytes([a[0] ^ a[1] ^ a[2] ^ a[3]]))
    if rx(1) != b"\x79":
        raise RuntimeError(f"address NACK at 0x{addr:08X}")

    # send nuber of bytes to read
    # send xx xor + wait for ACK
    #set_debug_pin(False)
    tx(bytes([length - 1, (length - 1) ^ 0xFF]))
    set_debug_pin(True)
    if rx(1) != b"\x79":
        raise RuntimeError("length NACK")
    
    data = rx(length, timeout=1000)
    if len(data) != length:
        raise RuntimeError(f"short read at 0x{addr:08X}: {len(data)}/{length}")
    return data


def read_chunk_with_retry(addr, length):
    for attempt in range(RETRIES):
        try:
            return read_mem(addr, length)
        except RuntimeError as e:
            print(f"\n  retry {attempt + 1}/{RETRIES}: {e}")
            enter_bootloader()        # clean state, then try again
    raise RuntimeError(f"giving up at 0x{addr:08X}")

def get_product_id():
    # Get ID as a sanity check (expect 0462 for the L45x family)
    if cmd(0x02) == b"\x79":
        n = rx(1)[0]
        pid = rx(n + 1)
        rx(1)
        if debug:
            print("product ID:", pid.hex())

